# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'


client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('./data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {
            'role': 'system',
            'content': 'You are a helpful recruitment assistant who responds in a very concise manner.',
        },
        {
            'role': 'user',
            'content': (
                'Extract the company name, job role, and minimum years of experience required '
                'from this job posting. Use null if no years requirement is stated. Return only '
                'a JSON object with keys company, role, and years_experience_required.\n\n'
                f'{snippet_text}'
            ),
        },
    ]



def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [   
         {
            'role': 'system',
            'content': 'You are a helpful recruitment assistant who responds in a very concise manner.',
        }, 
        {
            'role': 'user',
            'content': (
                'Extract company, role, and minimum years of experience from each job posting. '
                'Use null if no years requirement is stated, 0 if no prior experience is required, '
                'and the lower value for a range. Return only a JSON object with keys company, '
                'role, and years_experience_required.\\n\\n'
                'Example 1 — Job posting: "Bluebird Labs seeks a Platform Engineer. '
                'At least four years of experience is required."'
            ),
        },
        {
            'role': 'assistant',
            'content': '{"company": "Bluebird Labs", "role": "Platform Engineer", "years_experience_required": 4}',
        },
        {
            'role': 'user',
            'content': 'Example 2 — Job posting: "Maple Street is hiring a Marketing Associate. No prior experience required."',
        },
        {
            'role': 'assistant',
            'content': '{"company": "Maple Street", "role": "Marketing Associate", "years_experience_required": 0}',
        },
        {
            'role': 'user',
            'content': 'Example 3 — Job posting: "Redwood Analytics seeks a Data Scientist with 3 to 5 years of experience."',
        },
        {
            'role': 'assistant',
            'content': '{"company": "Redwood Analytics", "role": "Data Scientist", "years_experience_required": 3}',
        },
        {
            'role': 'user',
            'content': f'Now extract the fields from this job posting:\\n{snippet_text}',
        },
    ]



def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            'role': 'system',
            'content': (
                'You are an expert recruiter extracting structured information from job postings. '
                'Return exactly one valid JSON object, with no markdown or extra text, matching this schema: '
                '{"company": string, "role": string, "years_experience_required": integer or null}. '
                'Use null when no years requirement is stated, 0 when no prior experience is required, '
                'and the lower bound when a range is given. Do not infer unstated requirements.'
            ),
        },
        {'role': 'user', 'content': snippet_text},
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [{
        'role': 'user',
        'content': (
            'Think step by step to identify the company, job role, and minimum years of experience '
            'required in this posting. Treat no prior experience as 0, use the lower bound for a '
            'range, and use null if no years requirement is stated. Then return only a JSON object '
            'with keys company, role, and years_experience_required, without including the reasoning. '
            f'Job posting: {snippet_text}'
        ),
    }]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

#result = STRATEGIES["zero_shot"](snippets[0]);
print(snippets[0])
result = STRATEGIES['zero_shot'](snippets[1]['snippet'])
print(result)

{'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
[{'role': 'system', 'content': 'You are a helpful recruitment assistant who responds in a very concise manner.'}, {'role': 'user', 'content': "Extract the company name, job role, and minimum years of experience required from this job posting. Use null if no years requirement is stated. Return only a JSON object with keys company, role, and years_experience_required.\n\nWe're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."}]


## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    cleaned = text.strip()
    if cleaned.startswith('```'):
        lines = cleaned.splitlines()[1:]
        if lines and lines[-1].strip() == '```':
            lines = lines[:-1]
        cleaned = '\n'.join(lines).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        for index, character in enumerate(cleaned):
            if character == '{':
                try:
                    parsed, _ = decoder.raw_decode(cleaned[index:])
                except json.JSONDecodeError:
                    continue
                if isinstance(parsed, dict):
                    return parsed
        return None

    return parsed if isinstance(parsed, dict) else None


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    snippet_text = snippet['snippet']
    messages = STRATEGIES[strategy_name](snippet_text)

    started_at = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency_s = time.perf_counter() - started_at

    if not response.choices:
        raise RuntimeError(f'No response choices returned for {strategy_name} / {snippet["id"]}.')
    raw_response = response.choices[0].message.content
    if raw_response is None:
        raise RuntimeError(f'No text response returned for {strategy_name} / {snippet["id"]}.')
    if response.usage is None:
        raise RuntimeError(f'Token usage missing for {strategy_name} / {snippet["id"]}; cannot calculate cost.')

    usage = response.usage
    cost_usd = (
        usage.prompt_tokens * RATES[MODEL]['in']
        + usage.completion_tokens * RATES[MODEL]['out']
    )

    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'snippet_text': snippet_text,
        'raw_response': raw_response,
        'extracted': parse_response(raw_response),
        'cost_usd': cost_usd,
        'latency_s': latency_s,
        'prompt_tokens': usage.prompt_tokens,
        'completion_tokens': usage.completion_tokens,
    }


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = [
        run_one(strategy_name, snippet)
        for snippet in snippets
        for strategy_name in STRATEGIES
    ]
    return await asyncio.gather(*tasks)

In [ ]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'few_shot',
 'snippet_id': 'j01',
 'snippet_text': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.',
 'raw_response': '{"company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5}',
 'extracted': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'cost_usd': 5.985e-05,
 'latency_s': 1.388594759000057,
 'prompt_tokens': 299,
 'completion_tokens': 25}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [7]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    if extracted is None:
        return 0

    fields = ('company', 'role', 'years_experience_required')
    matches = 0
    for field in fields:
        if field not in extracted or field not in gold:
            continue
        actual = extracted[field]
        expected = gold[field]
        if isinstance(actual, str):
            actual = actual.strip().casefold()
        if isinstance(expected, str):
            expected = expected.strip().casefold()
        matches += actual == expected
    return matches


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    gold_fields = {
        field: gold[field]
        for field in ('company', 'role', 'years_experience_required')
    }
    judge_input = {
        'job_posting': snippet_text,
        'gold': gold_fields,
        'extracted': extracted,
    }
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a strict evaluator of job-posting extractions. Compare the candidate '
                'extraction with the gold fields and the job posting. Award 4 if all three fields '
                'are correct; 3 if two are correct with no fabricated data; 2 if one is correct '
                'or a field is fabricated; 1 if none are correct or the extraction is null or '
                'unparsable. Accept equivalent wording, spelled-out numbers, and the minimum '
                'value from an experience range. Return only a JSON object in the form '
                '{"score": 1}, using an integer score from 1 to 4.'
            ),
        },
        {
            'role': 'user',
            'content': json.dumps(judge_input, ensure_ascii=False),
        },
    ]
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    if not response.choices:
        raise RuntimeError('The LLM judge returned no response choices.')
    judge_response = response.choices[0].message.content
    if judge_response is None:
        raise RuntimeError('The LLM judge returned no text response.')

    parsed = parse_response(judge_response)
    score = parsed.get('score') if parsed is not None else None
    if isinstance(score, bool) or not isinstance(score, int) or not 1 <= score <= 4:
        raise ValueError(f'Invalid LLM judge score: {judge_response!r}')
    return score

In [8]:
# Apply scoring to all 40 results
judge_scores = await asyncio.gather(*[
    score_llm_judge(
        result['snippet_text'],
        result['extracted'],
        golden[result['snippet_id']],
    )
    for result in results
])

scored = []
for result, judge_score in zip(results, judge_scores):
    gold = golden[result['snippet_id']]
    scored.append({
        **result,
        'accuracy': score_accuracy(result['extracted'], gold),
        'parse_success': result['extracted'] is not None,
        'llm_judge_score': judge_score,
    })
print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [10]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.9,1.0,4.0,0.000,1.387
few_shot,3.0,1.0,4.0,0.001,1.407
structured,2.9,1.0,4.0,0.000,1.326
zero_shot,2.8,1.0,3.9,0.000,1.469


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```